<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2D: *Spatial Join Reservoirs*
##### Version Number: 4.0
---
### Contents
> *Define Geometries*\
> *Spatial Join of Nearest Sampling Locations with Reservoir Datasets*\
> *Export File*
---
### Notes
- Integrate reservoir readings with sampling grid data via a buffered spatial join.
### Inputs
- Daily Weather Readings - `samples_weather_NDVI.csv` 
- Reservoir Data - `reservoirs.csv` (built in appendix H) 
---
### Outputs  
- `samples_res.csv` Sampling grid dataset merged with buffered reservoir readings.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

### Data Loading

In [3]:
samples = pd.read_csv("../data/processed/samples_fires.csv")
reservoirs = pd.read_csv("../data/processed/reservoirs.csv")

In [4]:
samples['Date'] = pd.to_datetime(samples['Date'])
reservoirs['Date'] = pd.to_datetime(reservoirs['Date'])

In [5]:
samples = samples.sort_values(
    by=["grid_id", "Date"]
)

### Subset main dataset to speed up spatial join.

In [6]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep].copy()

### Add geometry to sampling grid and reservoir datasets

In [7]:
join_samples['geometry'] = [Point(xy) for xy in zip(join_samples ['centroid_easting'], join_samples ['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

In [8]:
reservoirs['geometry'] = [Point(xy) for xy in zip(reservoirs['Longitude'], reservoirs['Latitude'])]
reservoir_gdf = gpd.GeoDataFrame(reservoirs, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
reservoir_gdf_projected = reservoir_gdf.to_crs(3310)

In [9]:
def add_reservoir_distance_features_by_year(
    samples_gdf,
    reservoir_gdf,
    year
):

    # Filter by year
    samples_year = samples_gdf[samples_gdf['Date'].dt.year == year]
    reservoirs_year = reservoir_gdf[reservoir_gdf['Date'].dt.year == year]

    # if no sample centroids or reservoirs, skip year
    if samples_year.empty or reservoirs_year.empty:
        return samples_gdf

    for dt in samples_year['Date'].unique():
        
        # Subset dataframes by date
        samples_today = samples_year[samples_year['Date'] == dt]
        reservoirs_today = reservoirs_year[reservoirs_year['Date'] == dt]

        # if either empty, skip to next date
        if reservoirs_today.empty or samples_today.empty:
            continue

        # Compute distances
        dist_matrix = np.array([
            reservoirs_today.geometry.distance(sample_geom)
            for sample_geom in samples_today.geometry
        ])

        # Assign distances to closest reservoir and avg distance to all reservoirs within grid
        mask = samples_gdf['Date'] == dt
        samples_gdf.loc[mask, 'dist_to_reservoir_same_day'] = dist_matrix.min(axis=1)
        samples_gdf.loc[mask, 'avg_dist_to_all_reservoirs_same_day'] = dist_matrix.mean(axis=1)

    return samples_gdf


In [10]:
premerged = samples_gdf.copy()  # Snapshot for post merge checks

years = sorted(samples_gdf['Date'].dt.year.unique())

for yr in years:
    samples_gdf = add_reservoir_distance_features_by_year(
        samples_gdf,
        reservoir_gdf_projected,
        year=yr
    )

In [11]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 7)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [12]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

### Rejoin with main dataset

In [13]:
# Merge on BOTH station and date
samples_res = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [14]:
post_merge_check(samples_res, samples)

Premerged shape:  (608880, 75)
Merged shape:  (608880, 77)


Duplicates before merge:  0


Duplicates after merge:  0


NA values before merge:  0
NA values after merge:  0


## Export File

In [15]:
samples_res.to_csv('../data/processed/samples_res_distance.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
